# 99 — Validation against the legacy products that went into the paper

Compares every regenerated product with its counterpart in
`../FINAL_PAPER_ANALYSIS/` (paths in `config.LEGACY_*`):

* stacked exposures ↔ `Linear_Composites_Instrument_Frame_Chaitanya/`
* HDR FITS ↔ `ExpNorm_HDR_Images_Paras/`, `LDIC_HDR_Images_Paras/`
* figures ↔ the PNGs referenced by the LaTeX source
* tables ↔ `data/*.csv`

Writes `products/validation_report.csv`.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
report = []
region = (slice(1500, 4000), slice(2500, 6000))     # central window, away from the zero-padded borders

# Stacked exposures
for e in config.INV_EXPOSURES:
    ours = fits.getdata(utils.stacked_filename(e), memmap=True)
    leg = fits.getdata(config.LEGACY_STACKED_DIR / f"Linear_Composite_{e}.fits", memmap=True)
    for i, pos in enumerate(config.POLARIZER_POSITIONS):
        report.append(dict(product=f"stacked_exp{e} pol{pos}", **utils.compare_arrays(ours[i], leg[i], region)))

# HDR products (full frame)
legacy_hdr = {"expnorm": (config.LEGACY_HDR_EXPNORM_DIR, "hdr_image_ExpNorm_pol{}.fits"),
              "ldic":    (config.LEGACY_HDR_LDIC_DIR,    "hdr_image_LDIC_pol{}.fits")}
for method, (d, pat) in legacy_hdr.items():
    for pos in config.POLARIZER_POSITIONS:
        ours = fits.getdata(utils.hdr_filename(method, pos))
        leg = fits.getdata(d / pat.format(pos))
        report.append(dict(product=f"hdr_{method} pol{pos}", **utils.compare_arrays(ours, leg)))

rep = pd.DataFrame(report)
print(rep.to_string(index=False, float_format=lambda x: f"{x:.6g}"))

In [ ]:
# Figures: correlation of the regenerated PNGs with the paper PNGs
figs = {"expnorm": "expnorm_hdr_solar_frame.png", "ldic": "ldic_hdr_solar_frame.png", "histogram": "histogram_normalized_exposures.png"}
for k, name in figs.items():
    c = utils.compare_images(config.FIGURES_DIR / name, config.LEGACY_FIGURES[k])
    report.append(dict(product=f"figure {name}", corr=c))
    print(f"{name:40s} corr = {c:.4f}")
rep = pd.DataFrame(report)
rep.to_csv(config.PRODUCTS_DIR / "validation_report.csv", index=False)

In [ ]:
# Tables
leg_meta = pd.read_csv(config.LEGACY_PAPER_METADATA_CSV); meta = pd.read_csv(config.PAPER_METADATA_CSV)
key = ["Filename"]
same = meta.sort_values(key).reset_index(drop=True).equals(leg_meta.sort_values(key).reset_index(drop=True))
print("paper metadata table identical (up to row order of equal-exposure frames):", same)
c = pd.read_csv(config.SUN_MOON_CENTERS_CSV).set_index("filename"); l = pd.read_csv(config.LEGACY_SUN_MOON_CENTERS_CSV).set_index("filename")
j = c.join(l, rsuffix="_legacy", how="inner")
print("sun centre max |diff| (px):", float((j.sun_xc - j.sun_xc_legacy).abs().max()), float((j.sun_yc - j.sun_yc_legacy).abs().max()))

In [ ]:
# Side-by-side: regenerated vs paper figure
from PIL import Image
fig, axes = plt.subplots(2, 2, figsize=(12, 15))
for row, k in enumerate(("expnorm", "ldic")):
    axes[row, 0].imshow(Image.open(config.FIGURES_DIR / figs[k])); axes[row, 0].set_title(f"regenerated {figs[k]}")
    axes[row, 1].imshow(Image.open(config.LEGACY_FIGURES[k]));    axes[row, 1].set_title("paper")
for ax in axes.flat: ax.axis("off")
plt.tight_layout()